# 8. DCGAN: generate synthetic handwritten digits
Train a small Deep Convolutional GAN on MNIST. For a quick class demo, 3-5 epochs may be enough to show the idea, though image quality improves with more training.

In [ ]:
!pip -q install torch torchvision matplotlib

In [ ]:
import torch, torch.nn as nn, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
ds=datasets.MNIST("/content/data",train=True,download=True,transform=transforms.Compose([
    transforms.ToTensor(), transforms.Normalize((.5,),(.5,))]))
loader=DataLoader(ds,batch_size=128,shuffle=True,num_workers=2,drop_last=True)
print("Device:",device,"| dataset size:",len(ds))

In [ ]:
zdim=100
G=nn.Sequential(nn.Linear(zdim,256*7*7),nn.BatchNorm1d(256*7*7),nn.ReLU(True),
    nn.Unflatten(1,(256,7,7)),nn.ConvTranspose2d(256,128,4,2,1),nn.BatchNorm2d(128),nn.ReLU(True),
    nn.ConvTranspose2d(128,64,4,2,1),nn.BatchNorm2d(64),nn.ReLU(True),
    nn.Conv2d(64,1,3,1,1),nn.Tanh()).to(device)
D=nn.Sequential(nn.Conv2d(1,64,4,2,1),nn.LeakyReLU(.2,True),
    nn.Conv2d(64,128,4,2,1),nn.BatchNorm2d(128),nn.LeakyReLU(.2,True),
    nn.Flatten(),nn.Linear(128*7*7,1)).to(device)
optG=torch.optim.Adam(G.parameters(),lr=2e-4,betas=(.5,.999))
optD=torch.optim.Adam(D.parameters(),lr=2e-4,betas=(.5,.999))
criterion=nn.BCEWithLogitsLoss()
fixed=torch.randn(25,zdim,device=device)
epochs=3  # Use Colab GPU; increase to 10+ for more convincing results
for epoch in range(epochs):
    for real,_ in loader:
        real=real.to(device); n=real.size(0)
        # Train discriminator on real and generated images
        optD.zero_grad()
        loss_real=criterion(D(real),torch.ones(n,1,device=device))
        fake=G(torch.randn(n,zdim,device=device)).detach()
        loss_fake=criterion(D(fake),torch.zeros(n,1,device=device))
        lossD=loss_real+loss_fake; lossD.backward(); optD.step()
        # Train generator to fool discriminator
        optG.zero_grad()
        fake=G(torch.randn(n,zdim,device=device))
        lossG=criterion(D(fake),torch.ones(n,1,device=device))
        lossG.backward(); optG.step()
    print(f"Epoch {epoch+1}/{epochs} | D loss {lossD.item():.3f} | G loss {lossG.item():.3f}")

In [ ]:
G.eval()
with torch.no_grad(): samples=G(fixed).cpu()
fig,axes=plt.subplots(5,5,figsize=(6,6))
for ax,img in zip(axes.ravel(),samples):
    ax.imshow(img.squeeze(),cmap="gray",vmin=-1,vmax=1); ax.axis("off")
plt.suptitle("Synthetic MNIST-like samples"); plt.show()
torch.save(G.state_dict(),"/content/generator_mnist.pth")
print("Saved generator checkpoint to /content/generator_mnist.pth")

**Teaching point:** the generator makes samples from random noise; the discriminator learns to distinguish real training images from generated images. GAN training can oscillate and collapse, so output quality is not guaranteed. Never present synthetic outputs as real data.